# ELE 261 Mini Project — Automated Defect Detection (NEU Surface Defect Database)
Spatial filtering → Intensity enhancement → Frequency-domain processing → Classical features + ResNet18.

**Baseline (raw images) vs Proposed (preprocessed images).** Run the cells top to bottom (Runtime → Run all).

In [ ]:
!pip -q install kagglehub opencv-python scikit-image scikit-learn matplotlib

## 0. Imports & configuration

In [ ]:
import json
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

SEED = 42
IMG_SIZE = 200
EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
CODES = {"cr": "crazing", "in": "inclusion", "pa": "patches",
         "ps": "pitted_surface", "rs": "rolled-in_scale", "sc": "scratches"}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

## 1. Data loading & split

In [ ]:
def get_dataset_path(user_path=None):
    if user_path:
        return Path(user_path)
    import kagglehub
    return Path(kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database"))


def load_dataset(root, size=IMG_SIZE):
    """Reads every image under root. Label is taken from the file name (crazing_12.jpg -> crazing)."""
    imgs, names = [], []
    for p in sorted(Path(root).rglob("*")):
        if p.suffix.lower() not in EXTS:
            continue
        im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if im is None:
            continue
        base = p.stem.rsplit("_", 1)[0].lower()
        names.append(CODES.get(base, base))
        imgs.append(cv2.resize(im, (size, size)))
    classes = sorted(set(names))
    y = np.array([classes.index(n) for n in names])
    X = np.stack(imgs).astype(np.uint8)
    print(f"Loaded {len(X)} images, {len(classes)} classes: {classes}")
    return X, y, classes


def split_data(X, y, seed):
    from sklearn.model_selection import train_test_split
    idx = np.arange(len(X))
    tr, tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=seed)
    va, te = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=seed)
    return tr, va, te

## 2. A. Spatial-domain processing
Convolution, averaging, median, max, min, Laplacian sharpening, 1st/2nd derivatives.

In [ ]:
def to_f(img_u8):
    return img_u8.astype(np.float32) / 255.0


def to_u8(img):
    return np.clip(img * 255.0 + 0.5, 0, 255).astype(np.uint8)


def norm01(img):
    lo, hi = float(img.min()), float(img.max())
    return (img - lo) / (hi - lo + 1e-8)


def convolve(img, kernel):
    # filter2D computes correlation; flip the kernel to obtain true convolution
    return cv2.filter2D(img, -1, np.flipud(np.fliplr(kernel)).astype(np.float32),
                        borderType=cv2.BORDER_REFLECT)


def averaging_filter(img, k=5):
    return convolve(img, np.ones((k, k), np.float32) / (k * k))


def median_filter(img, k=3):
    return to_f(cv2.medianBlur(to_u8(img), k))


def maximum_filter(img, k=3):
    return cv2.dilate(img, np.ones((k, k), np.uint8))


def minimum_filter(img, k=3):
    return cv2.erode(img, np.ones((k, k), np.uint8))


LAPLACIAN_K = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32)


def laplacian(img):
    return convolve(img, LAPLACIAN_K)


def laplacian_sharpen(img, c=1.0):
    # centre coefficient is negative -> g = f - c * lap(f)
    return np.clip(img - c * laplacian(img), 0, 1)


def first_derivative(img):
    gx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=3)
    return np.sqrt(gx ** 2 + gy ** 2)


def second_derivative(img):
    return np.abs(laplacian(img))

## 3. B. Intensity enhancement
Gamma correction, contrast stretching, histogram equalization.

In [ ]:
def gamma_correction(img, gamma=0.7):
    return np.power(np.clip(img, 0, 1), gamma)


def contrast_stretch(img, p_low=1, p_high=99):
    lo, hi = np.percentile(img, [p_low, p_high])
    return np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)


def hist_equalization(img):
    return to_f(cv2.equalizeHist(to_u8(img)))

## 4. C. Frequency-domain processing
2D DFT / IDFT, spectrum, bands, Ideal/Butterworth/Gaussian LP & HP, noise removal (notch).

In [ ]:
def dist_grid(shape):
    M, N = shape
    u = np.arange(M) - M // 2
    v = np.arange(N) - N // 2
    V, U = np.meshgrid(v, u)
    return np.sqrt(U ** 2 + V ** 2).astype(np.float32)


def lowpass_tf(shape, d0, kind="gaussian", n=2):
    D = dist_grid(shape)
    if kind == "ideal":
        return (D <= d0).astype(np.float32)
    if kind == "butterworth":
        return 1.0 / (1.0 + (D / d0) ** (2 * n))
    if kind == "gaussian":
        return np.exp(-(D ** 2) / (2 * d0 ** 2)).astype(np.float32)
    raise ValueError(kind)


def highpass_tf(shape, d0, kind="gaussian", n=2):
    return 1.0 - lowpass_tf(shape, d0, kind, n)


def dft2(img):
    """Zero-padded (2M x 2N) centred 2-D DFT."""
    M, N = img.shape
    padded = np.zeros((2 * M, 2 * N), np.float32)
    padded[:M, :N] = img
    # centring by (-1)^(x+y) is equivalent to fftshift
    return np.fft.fftshift(np.fft.fft2(padded))


def idft2(F, shape):
    M, N = shape
    out = np.real(np.fft.ifft2(np.fft.ifftshift(F)))
    return out[:M, :N].astype(np.float32)


def fourier_spectrum(img):
    return np.log1p(np.abs(dft2(img)))


def freq_filter(img, make_tf):
    F = dft2(img)
    H = make_tf(F.shape)
    return idft2(F * H, img.shape)


def freq_lowpass(img, d0=40, kind="gaussian", n=2):
    return np.clip(freq_filter(img, lambda s: lowpass_tf(s, d0, kind, n)), 0, 1)


def freq_highpass(img, d0=30, kind="gaussian", n=2):
    return freq_filter(img, lambda s: highpass_tf(s, d0, kind, n))


def high_boost(img, d0=30, k=1.5, kind="gaussian"):
    """f + k * HP(f): emphasises scratches / crazing edges while keeping the background."""
    return np.clip(img + k * freq_highpass(img, d0, kind), 0, 1)


def radial_band_energy(img, bands=8):
    mag = fourier_spectrum(img)
    D = dist_grid(mag.shape)
    edges = np.linspace(0, D.max() / np.sqrt(2), bands + 1)
    return np.array([mag[(D >= edges[i]) & (D < edges[i + 1])].mean() for i in range(bands)])


# --- noise models + frequency-domain noise removal -------------------------------
def add_gaussian_noise(img, sigma=0.05, rng=None):
    rng = rng or np.random.default_rng(SEED)
    return np.clip(img + rng.normal(0, sigma, img.shape).astype(np.float32), 0, 1)


def add_salt_pepper(img, amount=0.03, rng=None):
    rng = rng or np.random.default_rng(SEED)
    out = img.copy()
    r = rng.random(img.shape)
    out[r < amount / 2] = 0.0
    out[r > 1 - amount / 2] = 1.0
    return out


def add_periodic_noise(img, amp=0.12, fx=0.12, fy=0.08):
    M, N = img.shape
    y, x = np.mgrid[0:M, 0:N]
    return np.clip(img + amp * np.sin(2 * np.pi * (fx * x + fy * y)), 0, 1).astype(np.float32)


def notch_reject(img, k_sigma=6.0, radius=12, width=3.0, max_peaks=6):
    """Automatic periodic-noise removal: detect isolated spectral spikes and notch them (with symmetric pair)."""
    F = dft2(img)
    mag = np.log1p(np.abs(F))
    D = dist_grid(F.shape)
    cand = np.where(D > radius, mag, 0)
    thr = cand[D > radius].mean() + k_sigma * cand[D > radius].std()
    ys, xs = np.where(cand > thr)
    order = np.argsort(-cand[ys, xs])[:max_peaks * 2]
    H = np.ones(F.shape, np.float32)
    cy, cx = F.shape[0] // 2, F.shape[1] // 2
    yy, xx = np.mgrid[0:F.shape[0], 0:F.shape[1]]
    for i in order:
        y0, x0 = ys[i], xs[i]
        for (py, px) in ((y0, x0), (2 * cy - y0, 2 * cx - x0)):
            H *= 1 - np.exp(-((yy - py) ** 2 + (xx - px) ** 2) / (2 * width ** 2))
    return np.clip(idft2(F * H, img.shape), 0, 1)

## 5. Proposed preprocessing pipeline
Median → contrast stretch → gamma → Gaussian high-boost (frequency domain).

In [ ]:
PRE_CHOICES = ["none", "stretch", "he", "proposed"]


def preprocess(img_u8, mode):
    f = to_f(img_u8)
    if mode == "none":
        return img_u8
    if mode == "stretch":
        return to_u8(contrast_stretch(f))
    if mode == "he":
        return to_u8(hist_equalization(f))
    if mode == "proposed":
        # 1) impulse-noise suppression  2) contrast stretching  3) gamma  4) Gaussian high-boost in frequency domain
        f = median_filter(f, 3)
        f = contrast_stretch(f, 1, 99)
        f = gamma_correction(f, 0.9)
        f = high_boost(f, d0=30, k=1.0, kind="gaussian")
        return to_u8(norm01(f))
    raise ValueError(mode)


def preprocess_all(X, mode):
    return np.stack([preprocess(im, mode) for im in X])

## 6. Figures for every technique

In [ ]:
def show_grid(images, titles, path, ncols=4, cmap="gray", figsize_per=3.0):
    n = len(images)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * figsize_per, nrows * figsize_per))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(im, cmap=cmap, vmin=None, vmax=None)
        ax.set_title(t, fontsize=9)
    plt.tight_layout()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=150)
    plt.close(fig)


def cmd_demo(X, y, classes, out):
    out = Path(out) / "figures"
    # one example per class
    samples = [(classes[c], to_f(X[np.where(y == c)[0][0]])) for c in range(len(classes))]
    name, img = samples[-1]  # scratches (or last class) for the detailed figures

    # --- spatial
    show_grid(
        [img, averaging_filter(img, 5), median_filter(img, 3), maximum_filter(img, 3),
         minimum_filter(img, 3), laplacian_sharpen(img), norm01(first_derivative(img)),
         norm01(second_derivative(img))],
        ["Original", "Averaging 5x5", "Median 3x3", "Maximum 3x3", "Minimum 3x3",
         "Laplacian sharpen", "1st derivative (Sobel)", "2nd derivative (|Laplacian|)"],
        out / f"spatial_{name}.png")

    # --- enhancement with histograms
    variants = [("Original", img), ("Gamma 0.5", gamma_correction(img, 0.5)),
                ("Gamma 1.5", gamma_correction(img, 1.5)), ("Contrast stretch", contrast_stretch(img)),
                ("Hist. equalization", hist_equalization(img))]
    fig, axes = plt.subplots(2, len(variants), figsize=(3 * len(variants), 6))
    for j, (t, im) in enumerate(variants):
        axes[0, j].imshow(im, cmap="gray", vmin=0, vmax=1)
        axes[0, j].set_title(t, fontsize=9)
        axes[0, j].axis("off")
        axes[1, j].hist(im.ravel(), bins=64, range=(0, 1))
    plt.tight_layout()
    plt.savefig(out / f"enhancement_{name}.png", dpi=150)
    plt.close(fig)

    # --- Fourier spectrum + filters
    show_grid([img, norm01(fourier_spectrum(img))], ["Original", "Fourier spectrum (log)"],
              out / f"spectrum_{name}.png", ncols=2)
    d0 = 40
    lp = [freq_lowpass(img, d0, k) for k in ("ideal", "butterworth", "gaussian")]
    hp = [norm01(freq_highpass(img, 30, k)) for k in ("ideal", "butterworth", "gaussian")]
    show_grid(lp + hp, [f"{k} LP (D0={d0})" for k in ("Ideal", "Butterworth", "Gaussian")] +
              [f"{k} HP (D0=30)" for k in ("Ideal", "Butterworth", "Gaussian")],
              out / f"freq_filters_{name}.png", ncols=3)

    # transfer functions
    shp = (2 * IMG_SIZE, 2 * IMG_SIZE)
    tfs = [lowpass_tf(shp, d0, k) for k in ("ideal", "butterworth", "gaussian")] + \
          [highpass_tf(shp, 30, k) for k in ("ideal", "butterworth", "gaussian")]
    show_grid(tfs, ["Ideal LP", "Butterworth LP", "Gaussian LP", "Ideal HP", "Butterworth HP", "Gaussian HP"],
              out / "transfer_functions.png", ncols=3)

    # frequency bands
    F = dft2(img)
    D = dist_grid(F.shape)
    bands = [(0, 10), (10, 40), (40, 100), (100, 1e9)]
    show_grid([np.clip(idft2(F * ((D >= a) & (D < b)), img.shape), -1, 1) for a, b in bands],
              [f"Band {a}-{int(min(b, 999))}" for a, b in bands], out / f"freq_bands_{name}.png", ncols=4)

    # noise removal
    per = add_periodic_noise(img)
    gau = add_gaussian_noise(img)
    sp = add_salt_pepper(img)
    show_grid([per, notch_reject(per), gau, freq_lowpass(gau, 50, "butterworth"), sp, median_filter(sp, 3)],
              ["Periodic noise", "Notch reject", "Gaussian noise", "Butterworth LP", "Salt & pepper", "Median 3x3"],
              out / f"noise_removal_{name}.png", ncols=2, figsize_per=3.2)

    # proposed pipeline across all classes
    rows = []
    for cname, im in samples:
        rows += [im, to_f(preprocess(to_u8(im), "proposed"))]
    show_grid(rows, sum([[f"{c} (raw)", f"{c} (proposed)"] for c, _ in samples], []),
              out / "proposed_pipeline_all_classes.png", ncols=4)
    print(f"Figures saved in {out}")

## 7. Denoising experiment (PSNR / SSIM)

In [ ]:
def cmd_denoise(X, out, n_images=60):
    import csv
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(X), min(n_images, len(X)), replace=False)
    noises = {
        "gaussian(s=0.05)": lambda im: add_gaussian_noise(im, 0.05, rng),
        "salt&pepper(3%)": lambda im: add_salt_pepper(im, 0.03, rng),
        "periodic": lambda im: add_periodic_noise(im),
    }
    filters = {
        "none": lambda im: im,
        "average5": lambda im: averaging_filter(im, 5),
        "median3": lambda im: median_filter(im, 3),
        "ideal_LP": lambda im: freq_lowpass(im, 50, "ideal"),
        "butterworth_LP": lambda im: freq_lowpass(im, 50, "butterworth"),
        "gaussian_LP": lambda im: freq_lowpass(im, 50, "gaussian"),
        "notch": lambda im: notch_reject(im),
    }
    rows = []
    for nname, nf in noises.items():
        for fname, ff in filters.items():
            p, s = [], []
            for i in idx:
                clean = to_f(X[i])
                den = ff(nf(clean))
                p.append(psnr(clean, den, data_range=1.0))
                s.append(ssim(clean, den, data_range=1.0))
            rows.append([nname, fname, np.mean(p), np.std(p), np.mean(s), np.std(s)])
            print(f"{nname:18s} {fname:15s} PSNR {np.mean(p):6.2f}  SSIM {np.mean(s):.3f}")
    Path(out).mkdir(parents=True, exist_ok=True)
    with open(Path(out) / "denoise_results.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["noise", "filter", "psnr_mean", "psnr_std", "ssim_mean", "ssim_std"])
        w.writerows(rows)

## 8. Classical features + SVM / Random Forest

In [ ]:
def handcrafted_features(img_u8):
    f = to_f(img_u8)
    hist = np.histogram(f, bins=16, range=(0, 1), density=True)[0]
    g = first_derivative(f)
    lap = second_derivative(f)
    stats = [f.mean(), f.std(), g.mean(), g.std(), lap.mean(), lap.std()]
    return np.concatenate([hist, stats, radial_band_energy(f, 8)])


def cmd_features(X, y, classes, out, seeds):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, f1_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import SVC
    results = {}
    for mode in PRE_CHOICES:
        Xp = preprocess_all(X, mode)
        feats = np.stack([handcrafted_features(im) for im in Xp])
        for mname, mk in {"SVM": lambda: make_pipeline(StandardScaler(), SVC(C=10, gamma="scale")),
                          "RF": lambda: RandomForestClassifier(300, random_state=SEED)}.items():
            accs, f1s = [], []
            for s in range(seeds):
                tr, va, te = split_data(X, y, SEED + s)
                clf = mk().fit(feats[np.r_[tr, va]], y[np.r_[tr, va]])
                pred = clf.predict(feats[te])
                accs.append(accuracy_score(y[te], pred))
                f1s.append(f1_score(y[te], pred, average="macro"))
            results[f"{mname}|{mode}"] = dict(acc=float(np.mean(accs)), acc_std=float(np.std(accs)),
                                              f1=float(np.mean(f1s)))
            print(f"{mname:4s} pre={mode:9s} acc {np.mean(accs):.4f} +- {np.std(accs):.4f}  F1 {np.mean(f1s):.4f}")
    Path(out).mkdir(parents=True, exist_ok=True)
    json.dump(results, open(Path(out) / "classical_results.json", "w"), indent=2)

## 9. ResNet18: baseline vs proposed

In [ ]:
def train_resnet(Xp, y, classes, seed, epochs, batch, lr, pretrained):
    import torch
    import torch.nn as nn
    import torch.nn.functional as Fn
    import torchvision
    from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

    set_seed(seed)
    torch.manual_seed(seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tr, va, te = split_data(Xp, y, seed)
    mean = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

    def prep(xb, augment=False):
        x = torch.from_numpy(xb).float().div(255).unsqueeze(1).to(dev)
        if augment:
            if random.random() < 0.5:
                x = torch.flip(x, [3])
            if random.random() < 0.5:
                x = torch.flip(x, [2])
        x = x.repeat(1, 3, 1, 1)
        x = Fn.interpolate(x, size=224, mode="bilinear", align_corners=False)
        return (x - mean) / std

    weights = torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
    model = torchvision.models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, len(classes))
    model.to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    yt = torch.from_numpy(y).long()

    def evaluate(idx):
        model.eval()
        preds = []
        with torch.no_grad():
            for i in range(0, len(idx), 128):
                b = idx[i:i + 128]
                preds.append(model(prep(Xp[b])).argmax(1).cpu())
        return torch.cat(preds).numpy()

    best_val, best_state = -1, None
    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(tr)
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]
            loss = Fn.cross_entropy(model(prep(Xp[b], True)), yt[b].to(dev))
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
        v = accuracy_score(y[va], evaluate(va))
        if v > best_val:
            best_val = v
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
    model.load_state_dict(best_state)
    pred = evaluate(te)
    return dict(acc=accuracy_score(y[te], pred), f1=f1_score(y[te], pred, average="macro"),
                val=best_val, cm=confusion_matrix(y[te], pred).tolist())


def cmd_train(X, y, classes, out, seeds, epochs, batch, lr, pretrained, modes):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    summary = {}
    for mode in modes:
        Xp = preprocess_all(X, mode)
        runs = []
        for s in range(seeds):
            r = train_resnet(Xp, y, classes, SEED + s, epochs, batch, lr, pretrained)
            runs.append(r)
            print(f"[ResNet18] pre={mode:9s} seed={SEED + s} acc={r['acc']:.4f} F1={r['f1']:.4f}")
        accs = [r["acc"] for r in runs]
        f1s = [r["f1"] for r in runs]
        cm = np.sum([np.array(r["cm"]) for r in runs], axis=0)
        summary[mode] = dict(acc_mean=float(np.mean(accs)), acc_std=float(np.std(accs)),
                             f1_mean=float(np.mean(f1s)), runs=[{k: v for k, v in r.items() if k != "cm"} for r in runs])
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right")
        ax.set_yticks(range(len(classes)), classes)
        for i in range(len(classes)):
            for j in range(len(classes)):
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
        ax.set_title(f"Confusion matrix - ResNet18 ({mode}), summed over seeds")
        plt.tight_layout()
        plt.savefig(out / f"confusion_{mode}.png", dpi=150)
        plt.close(fig)
    json.dump(summary, open(out / "resnet_results.json", "w"), indent=2)
    print("\nMode        Acc (mean +- std)     Macro-F1")
    for m, s in summary.items():
        print(f"{m:10s}  {s['acc_mean']:.4f} +- {s['acc_std']:.4f}    {s['f1_mean']:.4f}")


# ----------------------------------------------------------------------------

## 10. Run

In [ ]:
# الداتا بتتنزل أوتوماتيك. لو فشل التنزيل أو عندك فولدر جاهز، حط مساره هنا بدل None.
DATA_PATH = None   # مثال: "/content/NEU-DET"  (فولدر فيه الصور، حتى لو جوه فولدرات فرعية)

if DATA_PATH is None:
    import kagglehub
    DATA_PATH = kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database")
print("Path to dataset files:", DATA_PATH)
X, y, classes = load_dataset(DATA_PATH)
OUT = "results"

### 10.1 Figures (spatial / enhancement / frequency / proposed pipeline)

In [ ]:
cmd_demo(X, y, classes, OUT)
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f"{OUT}/figures/*.png")):
    print(f); display(Image(f, width=800))

### 10.2 Noise-removal comparison (PSNR / SSIM)

In [ ]:
cmd_denoise(X, OUT)

### 10.3 Classical features: raw vs preprocessed

In [ ]:
cmd_features(X, y, classes, OUT, seeds=3)

### 10.4 ResNet18: raw vs preprocessed
Use a GPU runtime (Runtime → Change runtime type → T4). Set `EPOCHS=1, SEEDS=1` first as a quick test.

In [ ]:
EPOCHS, SEEDS = 15, 3
cmd_train(X, y, classes, OUT, seeds=SEEDS, epochs=EPOCHS, batch=32, lr=3e-4,
          pretrained=True, modes=PRE_CHOICES)